# 4조 · 암 아형(SUBCLASS) 분류 · 최종 제출 코드

> **이 파일 하나로 학습부터 추론까지 실행돼요.** 통합 담당: ④ 박유정 · 최종 확인: ① 장성은
> 각 담당자는 자기 칸(【② 피처】【③ 모델】【④ 경량·앙상블】)에만 최종 코드를 넣어요. 칸 밖은 고치지 않아요.

## 제출 정보 (11/11 제출 전에 채우기)
| 항목 | 내용 |
|---|---|
| 팀 | 4조 (장성은, 홍시윤, 안주형, 박유정) |
| 개발 환경(OS) | 아래 "환경 정보" 셀 출력값을 옮겨 적기 |
| 라이브러리 버전 | 아래 "환경 정보" 셀 출력값 (requirements.txt와 같아야 함) |
| Random Seed | 42 (`SEED`) |
| 하이퍼파라미터 | 【③ 모델】 칸에 숫자로 기재 |
| 모델 파일 다운로드 링크 | (작성 예정) |
| 사전 학습 모델 | 사용 안 함 (사용 시 논문 링크와 출처 기재) |
| 외부 데이터 | 사용 안 함 |

## 데이터 누수 방지
- 모든 전처리·피처 선택·모델 학습은 train에만 fit 해요. test에는 transform과 predict만 해요.
- test의 통계, 분포, 행 개수, index는 쓰지 않아요.

## 1. 설정

In [ ]:
import os, random, platform
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "/data")   # 제출 규정: 데이터 입출력 경로 /data
OUTPUT_PATH = os.path.join(DATA_DIR, "submission.csv")
SEED = 42
N_SPLITS = 5
RUN_CV = False          # True면 교차검증 점수도 다시 계산해요 (시간이 오래 걸림)
TARGET, ID_COL = "SUBCLASS", "ID"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

## 2. 환경 정보 (출력값을 위 표에 옮겨 적어요)

In [ ]:
import sklearn, xgboost
print("OS:", platform.platform())
print("python:", platform.python_version())
print("numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

## 3. 데이터 불러오기

In [ ]:
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET)]
assert list(test.columns) == [ID_COL] + feature_cols   # 형식 확인만

X = train[feature_cols]
X_test = test[feature_cols]
le = LabelEncoder().fit(train[TARGET])                  # train 정답에만 fit
y = le.transform(train[TARGET])
print("train", X.shape, "| test", X_test.shape, "| classes", len(le.classes_))

## 4. 【② 피처】 전처리·피처 · 담당 ② 홍시윤
`src/preprocessing.py`, `src/features.py`의 최종 함수를 그대로 붙여 넣어요 (화요일 12시 확정본).

In [ ]:
# ===== 【② 피처】 시작 =====
"""
기본 전처리  ·  담당: ② 홍시윤 (수정하면 디스코드에 공유)

팀 기본 피처: WT 또는 결측 = 0, 변이 = 1 (11/06 킥오프에서 합의할 초안)

값의 형태 (train 기준)
- "WT" : 변이 없음 (Wild Type). 전체 값의 약 99%
- 그 외 : 변이 표기. 예) "R132H", "V600E". 여러 개면 공백으로 구분 "R273C R248Q"
- 결측 : 변이 정보 없음 → WT와 같게 처리 (고정 규칙, 통계 사용 아님)

이 파일의 변환은 "한 행 안에서만" 계산해서 누수가 없어요.
데이터로 학습하는 변환(인코더, 스케일러 등)을 추가할 때는 sklearn Transformer로 만들어
Pipeline 안에 넣어요 (fold 안에서만 fit 되도록).
"""
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

WT = "WT"


def binarize_mutations(X: pd.DataFrame) -> pd.DataFrame:
    """[팀 기본 피처] WT 또는 결측 = 0, 변이 = 1."""
    return (X.notna() & (X != WT)).astype(np.int8)


def basic_transformer() -> FunctionTransformer:
    """Pipeline에 넣어 쓰는 기본 전처리 (상태가 없어 fit해도 아무것도 배우지 않음)."""
    return FunctionTransformer(binarize_mutations)


"""
공통 피처  ·  담당: ② 홍시윤 (수정하면 디스코드에 공유)

여기에는 "효과가 검증된" 피처만 올려요. 실험 중인 피처는 각자 노트북에서 만들어 보고,
run_cv() 점수가 기본 피처보다 오르는 게 확인되면 이 파일로 옮겨요. (화요일 12시 확정)

규칙
- 한 행 안에서만 계산하는 피처는 여기 함수로 (train/test 통계를 쓰지 않으니 누수 없음)
- 피처 선택처럼 데이터로 학습하는 것은 sklearn Transformer로 만들어 Pipeline 안에 넣기
"""
import pandas as pd



def mutation_count(X: pd.DataFrame) -> pd.Series:
    """환자별 변이가 있는 유전자 개수."""
    return binarize_mutations(X).sum(axis=1).rename("n_mutated_genes")

# ===== 【② 피처】 끝 =====

## 5. 【③ 모델】 모델 정의 · 담당 ③ 안주형
최종 모델과 하이퍼파라미터를 숫자로 적어요. 지금은 베이스라인(CV 0.3209)이 들어 있어요.

In [ ]:
# ===== 【③ 모델】 시작 =====
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

XGB_PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6,
                  tree_method="hist", random_state=SEED, n_jobs=-1)

def make_model():
    return Pipeline([
        ("features", basic_transformer()),
        ("model", XGBClassifier(**XGB_PARAMS)),
    ])

MODELS = {"xgb_baseline": make_model}   # 앙상블이면 여기에 모델을 추가해요
# ===== 【③ 모델】 끝 =====

## 6. (선택) 교차검증 · 공통 검증 방식과 같아요

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def run_cv(model, X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    scores = []
    for tr, va in skf.split(np.zeros(len(y)), y):
        m = clone(model).fit(X.iloc[tr], y[tr])           # fold의 학습 부분에만 fit
        scores.append(f1_score(y[va], m.predict(X.iloc[va]), average="macro"))
    print(f"CV Macro F1 = {np.mean(scores):.4f} (std {np.std(scores):.4f})")
    return scores

if RUN_CV:
    for name, make in MODELS.items():
        print(name); run_cv(make(), X, y)

## 7. 전체 train으로 학습 → test 예측 확률

In [ ]:
test_probas = {}
for name, make in MODELS.items():
    model = make().fit(X, y)                 # train 전체에만 fit
    test_probas[name] = model.predict_proba(X_test)
    print(name, "학습 완료")

## 8. 【④ 경량·앙상블】 최종 예측 결합 · 담당 ④ 박유정
모델이 하나면 그대로, 여러 개면 05_ensemble에서 정한 가중치로 평균해요.

In [ ]:
# ===== 【④ 경량·앙상블】 시작 =====
ENSEMBLE_WEIGHTS = None   # 예) {"xgb_baseline": 0.6, "logreg": 0.4}  (None이면 동일 가중)

names = list(test_probas)
w = (np.ones(len(names)) / len(names) if ENSEMBLE_WEIGHTS is None
     else np.array([ENSEMBLE_WEIGHTS[n] for n in names]) / sum(ENSEMBLE_WEIGHTS.values()))
final_proba = sum(wi * test_probas[n] for wi, n in zip(w, names))
# ===== 【④ 경량·앙상블】 끝 =====

## 9. 제출 파일 저장

In [ ]:
sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert len(sub) == len(test) and (sub[ID_COL].values == test[ID_COL].values).all()
sub[TARGET] = le.inverse_transform(final_proba.argmax(axis=1))
sub.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUTPUT_PATH, sub.shape)
sub.head()